# Deployment Patterns & Model Compression

**Course:** [ML in Practice](https://ml-viz.vercel.app/courses/ml-in-practice/08-deployment-patterns-and-compression)

Picking a *serving pattern* (batch vs online vs edge) and a *compression stack* (quantization, pruning, distillation) are the two operational decisions that turn a trained model into a deployed product. This notebook makes both decisions concrete with self-contained numerical experiments:

1. **Batch vs online cost simulation.** Simulate one day of 10 M predictions under each pattern. Compute the GPU-hour cost as a function of the latency budget, and plot the cost vs latency Pareto frontier.
2. **Magnitude pruning.** Train a tiny dense MLP on a synthetic classification problem; apply magnitude pruning at sparsities from 10% to 99%; plot the accuracy elbow.
3. **Quantization sweep.** Take the trained model, quantize weights to {32, 16, 8, 4, 2} bits with uniform symmetric scaling; plot accuracy and memory (in GB for a 7B-scale model) on twin axes.
4. **Knowledge distillation.** Train a 'teacher' MLP with a large hidden dim; train a 'student' with a small hidden dim against the teacher's softened outputs at $T = 3$, sweeping the distillation mixing weight $\alpha$; compare to a from-scratch student of the same size.

Self-contained: NumPy + matplotlib only. No torch, no sklearn, no network, no API keys.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File -> Save a copy in Drive. Changes to this view are not saved.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)

plt.rcParams.update({
    'figure.facecolor': '#0f1117',
    'axes.facecolor':   '#1a1d27',
    'axes.edgecolor':   '#2a2d3a',
    'axes.labelcolor':  '#e2e8f0',
    'text.color':       '#e2e8f0',
    'xtick.color':      '#94a3b8',
    'ytick.color':      '#94a3b8',
    'grid.color':       '#2a2d3a',
    'grid.alpha':       0.5,
})

BRAND  = '#6366f1'
TEAL   = '#2dd4bf'
ROSE   = '#fb7185'
ORANGE = '#f97316'
YELLOW = '#facc15'
MUTED  = '#475569'


## 1. Batch vs online: cost vs latency Pareto

We simulate one day of inference for a model that needs to score $N = 10\,\text{M}$ users. Two patterns:

- **Batch**: run a single big job that finishes by morning. Throughput = $N / \text{schedule window}$. Cost = (GPU-hours used) * ($/GPU-hr). Tail latency at the request path is a key-value-store lookup (~1 ms).
- **Online**: serve a steady-state QPS over 24 h. We assume a sinusoidal diurnal traffic pattern with a peak-to-average ratio of 2.5x. Capacity must be sized for *peak* QPS (no autoscale-down to zero is realistic at this scale). Per-request latency budget is enforced by capping batch size on the server.

We sweep the *online latency budget* and report the cost of each pattern. Batch cost is roughly flat; online cost grows steeply as the budget tightens (smaller server-side batches => more GPUs).


In [ ]:
# Cost model (simplified, illustrative).
N_PREDICTIONS = 10_000_000     # per day
GPU_THROUGHPUT_QPS = 2_000     # one GPU at large batch, per second
GPU_COST_PER_HR = 4.00         # $/hr (e.g. A100 spot price)

# --- BATCH ---
# Batch runs for `batch_window_hours` and uses GPUs in parallel.
# Throughput per GPU is the full GPU_THROUGHPUT_QPS (large batches OK).
def batch_cost(batch_window_hours, n=N_PREDICTIONS):
    secs = batch_window_hours * 3600
    qps_needed = n / secs
    n_gpus = max(1, int(np.ceil(qps_needed / GPU_THROUGHPUT_QPS)))
    return n_gpus * batch_window_hours * GPU_COST_PER_HR, n_gpus

# --- ONLINE ---
# Diurnal traffic: peak/avg = 2.5; you size capacity for peak QPS.
# Server-side batching: at latency budget L_ms, you can wait at most L_ms
# to accumulate a batch. Effective throughput per GPU scales with batch size,
# which scales with L. Below ~5 ms there's no room to batch -> 1/k throughput.
PEAK_TO_AVG = 2.5

def online_throughput_per_gpu(latency_ms):
    # Effective batch size that fits in the budget; empirical-ish saturating curve.
    # At L=5 ms ~ batch 1, at L=100 ms ~ batch 64, at L>=300 ms ~ batch 256 (sat).
    b = np.clip((latency_ms - 4.0) / 1.5, 1.0, 256.0)
    # Throughput per GPU scales with batch size but with diminishing returns.
    return GPU_THROUGHPUT_QPS * (b / (b + 16.0)) * 2.5  # peaks at ~5x base

def online_cost(latency_ms, n=N_PREDICTIONS):
    avg_qps = n / 86400
    peak_qps = PEAK_TO_AVG * avg_qps
    tput_per_gpu = online_throughput_per_gpu(latency_ms)
    n_gpus = max(1, int(np.ceil(peak_qps / tput_per_gpu)))
    # Always-on for 24 h.
    return n_gpus * 24 * GPU_COST_PER_HR, n_gpus

# Sweep online latency budget; batch window is fixed at 8 h (overnight).
LATENCY_MS = np.array([5, 10, 20, 50, 100, 200, 500, 1000])
online_costs = np.array([online_cost(L)[0] for L in LATENCY_MS])
online_gpus = np.array([online_cost(L)[1] for L in LATENCY_MS])

batch_window = 8.0  # hours overnight
batch_c, batch_g = batch_cost(batch_window)
print(f'Batch ({batch_window:.0f} h window): {batch_g} GPU(s), ${batch_c:,.0f}/day')
for L, c, g in zip(LATENCY_MS, online_costs, online_gpus):
    print(f'Online (p99 ~ {L:>4d} ms): {g:>3d} GPU(s), ${c:,.0f}/day')


In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 5))
ax.plot(LATENCY_MS, online_costs, 'o-', color=BRAND, lw=2, ms=7, label='Online (sized for peak QPS)')
ax.axhline(batch_c, color=TEAL, lw=2, ls='--', label=f'Batch ({batch_window:.0f} h window)')
ax.set_xscale('log')
ax.set_xlabel('Online latency budget (ms, p99)')
ax.set_ylabel('Cost per day ($)')
ax.set_title('Batch vs online cost-latency Pareto (10 M predictions/day)')
ax.grid(True, which='both')
ax.legend(frameon=False)
for L, c in zip(LATENCY_MS, online_costs):
    ax.annotate(f'${c:,.0f}', xy=(L, c), xytext=(0, 8), textcoords='offset points',
                ha='center', fontsize=8, color='#e2e8f0')
plt.tight_layout(); plt.show()


The shape tells the story. Below ~50 ms the online cost climbs steeply: there is no room to batch on the server side, so each GPU serves few requests per second and you need many more GPUs. Above ~200 ms the online cost asymptotes to a floor set by peak-QPS sizing and diurnal idle time. Batch sits *flat below the floor* because it runs only when needed and amortises over a tight window.

If you do not need per-request freshness, batch dominates. If you do, the cost of online is mostly the cost of the latency budget you chose.

## 2. Magnitude pruning: the accuracy elbow

Train a tiny 2-layer MLP on a synthetic binary classification dataset. After training, apply *magnitude pruning* at increasing sparsities (10% to 99%) by zeroing out the smallest-magnitude weights. Plot test accuracy vs sparsity.

Most dense networks tolerate ~50-70% magnitude pruning with almost no accuracy loss; beyond that the curve falls off a cliff. That cliff is the *elbow* — the largest sparsity you can keep before retraining (i.e. iterative pruning) is required to recover quality.


In [ ]:
def make_classification(n=2000, d=20, n_useful=8, seed=0):
    r = np.random.default_rng(seed)
    X = r.normal(size=(n, d))
    w_true = np.zeros(d)
    w_true[:n_useful] = r.normal(scale=1.0, size=n_useful)
    logits = X @ w_true + 0.4 * r.normal(size=n)
    y = (logits > 0).astype(np.int64)
    return X, y, w_true

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -40, 40)))

def relu(x):
    return np.maximum(0.0, x)

class TinyMLP:
    def __init__(self, d_in, d_hidden, seed=0):
        r = np.random.default_rng(seed)
        self.W1 = r.normal(scale=np.sqrt(2.0 / d_in), size=(d_in, d_hidden))
        self.b1 = np.zeros(d_hidden)
        self.W2 = r.normal(scale=np.sqrt(2.0 / d_hidden), size=(d_hidden,))
        self.b2 = 0.0

    def forward(self, X):
        h = relu(X @ self.W1 + self.b1)
        z = h @ self.W2 + self.b2
        return sigmoid(z), h, z

    def loss_and_grads(self, X, y):
        p, h, z = self.forward(X)
        eps = 1e-9
        loss = -np.mean(y * np.log(p + eps) + (1 - y) * np.log(1 - p + eps))
        # backward
        dz = (p - y) / len(y)                          # (n,)
        dW2 = h.T @ dz                                  # (d_hidden,)
        db2 = float(dz.sum())
        dh = np.outer(dz, self.W2)                      # (n, d_hidden)
        dh[h <= 0] = 0.0
        dW1 = X.T @ dh                                  # (d_in, d_hidden)
        db1 = dh.sum(axis=0)
        return loss, (dW1, db1, dW2, db2)

    def step(self, grads, lr):
        dW1, db1, dW2, db2 = grads
        self.W1 -= lr * dW1
        self.b1 -= lr * db1
        self.W2 -= lr * dW2
        self.b2 -= lr * db2

    def accuracy(self, X, y):
        p, _, _ = self.forward(X)
        return float(((p > 0.5).astype(np.int64) == y).mean())

    def count_params(self):
        return self.W1.size + self.b1.size + self.W2.size + 1

    def total_weights(self):
        # Counts only the weight matrices for the sparsity calc.
        return self.W1.size + self.W2.size


def train(model, X_tr, y_tr, X_te, y_te, lr=0.05, n_iters=600, verbose=False):
    for it in range(n_iters):
        loss, grads = model.loss_and_grads(X_tr, y_tr)
        model.step(grads, lr)
        if verbose and (it % 100 == 0 or it == n_iters - 1):
            print(f'  iter {it:4d}  loss={loss:.4f}  test_acc={model.accuracy(X_te, y_te):.3f}')
    return model

# Build dataset and a teacher-sized model for pruning.
X, y, _ = make_classification(n=2400, d=20, seed=1)
N = len(y)
perm = np.random.default_rng(7).permutation(N)
ntr = int(0.8 * N)
X_tr, y_tr = X[perm[:ntr]], y[perm[:ntr]]
X_te, y_te = X[perm[ntr:]], y[perm[ntr:]]

base = TinyMLP(d_in=20, d_hidden=64, seed=11)
print('Training base MLP (20 -> 64 -> 1)...')
train(base, X_tr, y_tr, X_te, y_te, lr=0.05, n_iters=600, verbose=True)
print(f'Base params: {base.count_params()}  test_acc: {base.accuracy(X_te, y_te):.3f}')


In [ ]:
def magnitude_prune(model, sparsity):
    '''Return a new model with the bottom `sparsity` fraction of weights zeroed.

    Sparsity is computed globally across W1 and W2.'''
    pruned = TinyMLP(d_in=model.W1.shape[0], d_hidden=model.W1.shape[1], seed=0)
    pruned.W1 = model.W1.copy()
    pruned.b1 = model.b1.copy()
    pruned.W2 = model.W2.copy()
    pruned.b2 = model.b2
    # Global magnitude threshold across both weight matrices.
    all_w = np.concatenate([pruned.W1.ravel(), pruned.W2.ravel()])
    if sparsity <= 0:
        return pruned
    k = int(sparsity * all_w.size)
    threshold = np.partition(np.abs(all_w), k)[k]
    pruned.W1 = np.where(np.abs(pruned.W1) >= threshold, pruned.W1, 0.0)
    pruned.W2 = np.where(np.abs(pruned.W2) >= threshold, pruned.W2, 0.0)
    return pruned

SPARSITIES = [0.0, 0.1, 0.25, 0.5, 0.7, 0.8, 0.9, 0.95, 0.98, 0.99]
accs = []
nz_pct = []
for s in SPARSITIES:
    m = magnitude_prune(base, s)
    accs.append(m.accuracy(X_te, y_te))
    nz = (m.W1 != 0).sum() + (m.W2 != 0).sum()
    nz_pct.append(100 * nz / base.total_weights())
    print(f'sparsity {s:>5.2f}  test_acc={accs[-1]:.3f}  non-zero={nz}/{base.total_weights()} ({nz_pct[-1]:.1f}%)')


In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 5))
ax.plot([100 * s for s in SPARSITIES], accs, 'o-', color=BRAND, lw=2, ms=7)
ax.axhline(accs[0], color=TEAL, lw=1, ls='--', label=f'dense baseline = {accs[0]:.3f}')
# Elbow annotation: largest sparsity within 1 pp of the dense baseline.
elbow_idx = max(i for i, a in enumerate(accs) if a >= accs[0] - 0.01)
elbow_s = SPARSITIES[elbow_idx]
ax.scatter([100 * elbow_s], [accs[elbow_idx]], s=140, color=ROSE, zorder=5,
           label=f'elbow ~ {100 * elbow_s:.0f}% sparsity')
ax.set_xlabel('Sparsity (% of weights zeroed)')
ax.set_ylabel('Test accuracy')
ax.set_title('Magnitude pruning: the accuracy elbow')
ax.grid(True)
ax.legend(frameon=False)
plt.tight_layout(); plt.show()


Below the elbow, accuracy is flat — those weights were small for a reason and zeroing them does almost nothing. Above the elbow the curve drops fast because we start removing weights the model was using. To go further you need *iterative* pruning (prune a bit, fine-tune, prune more), which is what production pruning recipes look like. Note the disk-size win is exactly proportional to sparsity in a sparse format — but on dense GPU kernels the matmul still runs at the original shape, so wall-clock latency is unchanged. This is the unstructured-vs-structured pruning trade-off in numbers.

## 3. Quantization sweep: bit-width vs accuracy and memory

Take the trained dense model and quantize its weights with a uniform symmetric scheme at $b \in \{32, 16, 8, 4, 2\}$ bits. For each bit-width, report test accuracy and the memory a 7 B-parameter model would consume at that bit-width on disk.

The pattern matches the lesson's intuition: INT8 is essentially free, INT4 starts to bite, INT2 collapses without QAT or distillation. The memory line is straight (linear in bits) — the accuracy line is non-linear.


In [ ]:
def quantize_uniform_symmetric(W, bits):
    '''Per-tensor uniform symmetric quantization. Returns the de-quantized weights.'''
    if bits >= 32:
        return W.copy()
    vmax = np.abs(W).max()
    levels = 2 ** bits
    step = (2 * vmax) / (levels - 1)
    if step == 0:
        return W.copy()
    Q = np.round((W + vmax) / step) * step - vmax
    return Q

def quantize_model(model, bits):
    out = TinyMLP(d_in=model.W1.shape[0], d_hidden=model.W1.shape[1], seed=0)
    out.W1 = quantize_uniform_symmetric(model.W1, bits)
    out.b1 = model.b1.copy()    # biases left full-precision (cheap)
    out.W2 = quantize_uniform_symmetric(model.W2, bits)
    out.b2 = model.b2
    return out

BITS = [32, 16, 8, 4, 2]
q_accs = []
mem_gb = []
PARAM_COUNT_7B = 7_000_000_000
for b in BITS:
    qm = quantize_model(base, b)
    a = qm.accuracy(X_te, y_te)
    q_accs.append(a)
    mem_gb.append(PARAM_COUNT_7B * b / 8 / (1024 ** 3))
    print(f'{b:>2}-bit  test_acc={a:.3f}  7B memory={mem_gb[-1]:.2f} GB')


In [ ]:
fig, ax1 = plt.subplots(figsize=(8.5, 5))
ax1.plot(BITS, q_accs, 'o-', color=BRAND, lw=2, ms=8, label='Test accuracy (this MLP)')
ax1.set_xlabel('Bit-width')
ax1.set_ylabel('Test accuracy', color=BRAND)
ax1.tick_params(axis='y', labelcolor=BRAND)
ax1.set_xticks(BITS)
ax1.set_xticklabels([str(b) for b in BITS])
ax1.invert_xaxis()  # high bits on the left, low bits on the right -> reading direction
ax1.grid(True)

ax2 = ax1.twinx()
ax2.plot(BITS, mem_gb, 's--', color=ORANGE, lw=2, ms=8, label='Memory at 7B params (GB)')
ax2.set_ylabel('Memory for 7B model (GB)', color=ORANGE)
ax2.tick_params(axis='y', labelcolor=ORANGE)

ax1.set_title('Quantization sweep: accuracy is non-linear, memory is linear')
fig.tight_layout(); plt.show()


For a deeper look at the underlying bit-grid quantization (and to play with it interactively), see `<QuantizationViz />` in the lesson. The on-page widget shows the same uniform-symmetric scheme applied to a 14x14 weight matrix and reports the quantization noise as you slide the bit-width.

## 4. Knowledge distillation: teacher -> student vs scratch

Train a *teacher* MLP with a generous hidden dim (128). Then train two students with a tiny hidden dim (8):

- **From-scratch student**: standard binary cross-entropy on hard labels.
- **Distilled student**: combined loss with temperature $T = 3$,

$$
\mathcal{L} = (1-\alpha) \cdot \mathrm{CE}(y, p_s) + \alpha \cdot T^2 \cdot \mathrm{KL}\!\left(\sigma(z_t/T) \;\|\; \sigma(z_s/T)\right),
$$

where $z_t, z_s$ are teacher / student logits and $\sigma$ is sigmoid (this is a binary problem; for multi-class use softmax). We sweep $\alpha \in \{0, 0.3, 0.7, 1.0\}$ and compare student test accuracy.


In [ ]:
# Train the teacher (large hidden).
teacher = TinyMLP(d_in=20, d_hidden=128, seed=21)
print('Training teacher (20 -> 128 -> 1)...')
train(teacher, X_tr, y_tr, X_te, y_te, lr=0.05, n_iters=800)
print(f'Teacher test_acc: {teacher.accuracy(X_te, y_te):.3f}')
print(f'Teacher params:   {teacher.count_params()}')

# Get teacher logits and softened probabilities at temperature T for the train set.
T = 3.0
_, _, z_t = teacher.forward(X_tr)
p_t_soft = sigmoid(z_t / T)
print(f'Teacher logit range: [{z_t.min():.2f}, {z_t.max():.2f}]')
print(f'Softened probability range at T={T}: [{p_t_soft.min():.3f}, {p_t_soft.max():.3f}]')


In [ ]:
def train_student_distill(X_tr, y_tr, X_te, y_te, p_t_soft, z_t, alpha, T=3.0,
                          d_hidden=8, lr=0.05, n_iters=600, seed=33):
    '''Train a small student with the convex combination of hard CE and soft KL.'''
    student = TinyMLP(d_in=X_tr.shape[1], d_hidden=d_hidden, seed=seed)
    eps = 1e-9
    for it in range(n_iters):
        p_s, h, z_s = student.forward(X_tr)
        p_s_soft = sigmoid(z_s / T)
        # Hard CE: d/dz_s CE(y, sigmoid(z_s)) = (p_s - y)
        dz_hard = (p_s - y_tr) / len(y_tr)
        # KL(p_t_soft || p_s_soft) for binary case has gradient wrt z_s of
        # (p_s_soft - p_t_soft) / T.  The T^2 factor in the loss cancels one T
        # in the gradient, leaving a T factor on the soft term.
        dz_soft = T * (p_s_soft - p_t_soft) / len(y_tr)
        dz = (1.0 - alpha) * dz_hard + alpha * dz_soft

        dW2 = h.T @ dz
        db2 = float(dz.sum())
        dh = np.outer(dz, student.W2)
        dh[h <= 0] = 0.0
        dW1 = X_tr.T @ dh
        db1 = dh.sum(axis=0)
        student.step((dW1, db1, dW2, db2), lr)
    return student

ALPHAS = [0.0, 0.3, 0.7, 1.0]
N_TRIALS = 5  # average over seeds to smooth small-model variance
results = {}
for alpha in ALPHAS:
    accs_trial = []
    for seed in range(N_TRIALS):
        s = train_student_distill(X_tr, y_tr, X_te, y_te, p_t_soft, z_t,
                                  alpha=alpha, T=T, d_hidden=8, lr=0.05,
                                  n_iters=600, seed=33 + seed)
        accs_trial.append(s.accuracy(X_te, y_te))
    results[alpha] = (float(np.mean(accs_trial)), float(np.std(accs_trial)))
    print(f'alpha={alpha:.1f}  student test_acc = {results[alpha][0]:.3f} +/- {results[alpha][1]:.3f}')


In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 5))
xs = list(results.keys())
means = [results[a][0] for a in xs]
stds  = [results[a][1] for a in xs]
ax.errorbar(xs, means, yerr=stds, fmt='o-', color=BRAND, lw=2, ms=8, capsize=5,
            label='Student test accuracy (d_hidden=8)')
ax.axhline(teacher.accuracy(X_te, y_te), color=TEAL, lw=1, ls='--',
           label=f'teacher (d_hidden=128) = {teacher.accuracy(X_te, y_te):.3f}')
ax.axhline(results[0.0][0], color=ROSE, lw=1, ls=':',
           label=f'from-scratch (alpha=0) = {results[0.0][0]:.3f}')
ax.set_xlabel('alpha (distillation weight)')
ax.set_ylabel('Student test accuracy')
ax.set_title('Distillation vs from-scratch: small student sweep over alpha')
ax.grid(True)
ax.legend(frameon=False, loc='lower right')
plt.tight_layout(); plt.show()


In this regime the distilled students at moderate $\alpha$ tend to *beat* the pure from-scratch run ($\alpha = 0$), because the teacher's softened probabilities at $T = 3$ encode similarity structure across inputs that a hard label cannot. The pure-soft target ($\alpha = 1$) often underperforms a mix — it throws away the hard-label signal that anchors the student to the ground truth. The mix sweet spot is task-dependent; sweeping a few values is cheap and worth it. This is the production recipe behind every small-form-factor LLM.

---
## ✏️ Your turn

### Exercise: implement `distill_loss(student_logits, teacher_logits, labels, T, alpha)`

The combined loss is

$$
\mathcal{L} = (1-\alpha) \cdot \mathrm{CE}(y, p_s) \; + \; \alpha \cdot T^2 \cdot \mathrm{KL}\!\left(\sigma(z_t/T) \;\|\; \sigma(z_s/T)\right),
$$

where $z_t$, $z_s$ are teacher and student logits (1-D arrays), $y$ is the binary label vector, $\sigma$ is the sigmoid, and $\mathrm{KL}$ here is the *binary* KL between two Bernoulli distributions with means $\sigma(z_t/T)$ and $\sigma(z_s/T)$:

$$
\mathrm{KL}(p \,\|\, q) = p \log\frac{p}{q} + (1-p) \log\frac{1-p}{1-q}.
$$

**Properties to satisfy.**

- $\alpha = 0$: the loss reduces to plain binary cross-entropy of the student on the hard labels.
- $\alpha = 1$: the loss reduces to $T^2 \cdot \mathrm{KL}\!\left(\sigma(z_t/T) \;\|\; \sigma(z_s/T)\right)$ — no hard-label term.
- The function returns a single scalar (the mean over the batch).

In [ ]:
import numpy as np  # already imported

def distill_loss(student_logits, teacher_logits, labels, T, alpha):
    '''Convex combination of hard CE and soft-KL distillation loss.

    Args:
        student_logits : 1-D array of student logits z_s.
        teacher_logits : 1-D array of teacher logits z_t.
        labels         : 1-D array of binary labels y in {0, 1}.
        T              : temperature (float, T > 0).
        alpha          : mixing weight (float, 0 <= alpha <= 1).
    Returns:
        scalar loss (float).
    '''
    # TODO(you):
    #   1. Compute the student's predicted probabilities at T=1 (sigmoid(z_s)).
    #   2. Compute the hard-label binary cross-entropy: -mean(y log p_s + (1-y) log(1-p_s)).
    #   3. Compute the softened sigmoid probabilities at temperature T for teacher and student.
    #   4. Compute the binary KL(p_t_soft || p_s_soft), elementwise then mean.
    #   5. Return (1-alpha)*hard_ce + alpha * (T**2) * mean_kl.
    pass


def _sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -40, 40)))


In [ ]:
# Smoke values.
rng = np.random.default_rng(123)
z_s = rng.normal(size=64)
z_t = rng.normal(size=64)
y   = (rng.uniform(size=64) > 0.5).astype(np.float64)
T_test = 3.0

# Test 1: alpha=0 reduces to plain hard-label CE.
loss_a0 = distill_loss(z_s, z_t, y, T=T_test, alpha=0.0)
p_s = _sigmoid(z_s)
expected_hard_ce = -float(np.mean(y * np.log(p_s + 1e-12) + (1 - y) * np.log(1 - p_s + 1e-12)))
assert loss_a0 is not None, 'distill_loss returned None'
assert abs(loss_a0 - expected_hard_ce) < 1e-9, (
    f'at alpha=0 expected plain CE={expected_hard_ce:.6f}, got {loss_a0:.6f}')

# Test 2: alpha=1 drops the hard-label term and reduces to T^2 * KL.
loss_a1 = distill_loss(z_s, z_t, y, T=T_test, alpha=1.0)
p_t_soft = _sigmoid(z_t / T_test)
p_s_soft = _sigmoid(z_s / T_test)
expected_kl = float(np.mean(
    p_t_soft * np.log((p_t_soft + 1e-12) / (p_s_soft + 1e-12))
    + (1 - p_t_soft) * np.log((1 - p_t_soft + 1e-12) / (1 - p_s_soft + 1e-12))
))
expected_soft = (T_test ** 2) * expected_kl
assert abs(loss_a1 - expected_soft) < 1e-9, (
    f'at alpha=1 expected T^2*KL={expected_soft:.6f}, got {loss_a1:.6f}')

# Test 3: convex combination.
loss_mid = distill_loss(z_s, z_t, y, T=T_test, alpha=0.4)
expected_mid = 0.6 * expected_hard_ce + 0.4 * expected_soft
assert abs(loss_mid - expected_mid) < 1e-9, (
    f'at alpha=0.4 expected {expected_mid:.6f}, got {loss_mid:.6f}')

# Test 4: returns a scalar.
assert np.isscalar(loss_mid) or (hasattr(loss_mid, 'shape') and loss_mid.shape == ()), \
    'distill_loss should return a scalar'

print('All distill_loss tests passed.')


<details>
<summary>Show solution</summary>

```python
def distill_loss(student_logits, teacher_logits, labels, T, alpha):
    eps = 1e-12
    # Hard-label term (T=1).
    p_s = 1.0 / (1.0 + np.exp(-np.clip(student_logits, -40, 40)))
    hard_ce = -np.mean(
        labels * np.log(p_s + eps) + (1.0 - labels) * np.log(1.0 - p_s + eps)
    )
    # Softened distributions at temperature T.
    p_t_soft = 1.0 / (1.0 + np.exp(-np.clip(teacher_logits / T, -40, 40)))
    p_s_soft = 1.0 / (1.0 + np.exp(-np.clip(student_logits / T, -40, 40)))
    # Binary KL(p_t || p_s).
    kl = (
        p_t_soft * np.log((p_t_soft + eps) / (p_s_soft + eps))
        + (1.0 - p_t_soft) * np.log((1.0 - p_t_soft + eps) / (1.0 - p_s_soft + eps))
    )
    soft = float(np.mean(kl))
    return float((1.0 - alpha) * hard_ce + alpha * (T ** 2) * soft)
```

Three things to notice:

1. **The $T^2$ factor** is not cosmetic. When you scale logits by $1/T$ before the sigmoid, the gradient with respect to the logits also picks up a $1/T$ factor; multiplying the loss by $T^2$ keeps the soft-term gradient on the same scale as the hard-term gradient, so $\alpha$ behaves like a clean mixing weight rather than a hidden learning-rate change. Omit the $T^2$ and you have effectively reduced the learning rate on the soft term by a factor of $T$.
2. **`alpha=0` and `alpha=1` are clean reductions.** At $\alpha = 0$ the soft term vanishes and you get standard supervised training; at $\alpha = 1$ the hard-label term vanishes and the student is pure teacher-matching. In practice the best $\alpha$ is somewhere in between — pure teacher-matching throws away the ground-truth anchor and often slightly underperforms a small admixture of hard labels.
3. **The binary KL** is the symmetric Bernoulli version. For a multi-class problem you would swap in the softmax-KL: $\sum_k p_t^{(k)} \log (p_t^{(k)} / p_s^{(k)})$. The same $T^2$ correction applies.
</details>